# Example 15.3 / 1: Ask source-grounded questions

Companion notebook for **15.3 AMPL in NotebookLM** in the AMPL lab Documentation, section "Ask source-grounded questions". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Ask source-grounded questions

To check the answer, open each citation, then run the code it gives with the model below in
place of your own. The example below runs both cases so you can compare them with the answer:
`ampl.set_data(df, "PROD")` fills the set from the DataFrame index and each parameter from the
column with the same name, and a two-index parameter takes a Series whose index has two levels.
If the suggested code prints the same two dictionaries, it loads the data correctly:

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
set PROD;                         # steel products
set WEEKS;                        # planning weeks
param rate {PROD} > 0;            # tons per hour
param profit {PROD};              # dollars per ton
param market {PROD, WEEKS} >= 0;  # weekly maximum sales, tons

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [4]:
# data.py
import pandas as pd

products = pd.DataFrame(
    {"rate": [200, 140], "profit": [25, 30]},
    index=pd.Index(["bands", "coils"], name="PROD"),
)
ampl.set_data(products, "PROD")

ampl.set["WEEKS"] = [1, 2]
market = pd.DataFrame(
    {"PROD": ["bands", "bands", "coils", "coils"],
     "WEEKS": [1, 2, 1, 2],
     "market": [6000, 6500, 4000, 4200]}
).set_index(["PROD", "WEEKS"])
ampl.param["market"] = market["market"]

In [5]:
# run.py
print(ampl.param["rate"].to_dict())
print(ampl.param["market"].to_dict())

{'bands': 200, 'coils': 140}
{('bands', 1): 6000, ('bands', 2): 6500, ('coils', 1): 4000, ('coils', 2): 4200}
